# Semantic scheduler — real-data supervised D training
CPU managed Kaggle Torch, NumPy and LightGBM; source is canonical WSL snapshot. Ten supervised architectures, three seeds, max50epochs with patience8,64generated training episodes/seed and512optimization snapshots. Declared D budgets use original VLM silver tiers separate from SPI predictions. Validation is scheduler-only grouped development within exposed source-train; source test unopened. No independent emergency accuracy claim. This version does not claim RL or new contextual VLM training; those remain subsequent suite increments. Relevance is unavailable in this offline CPU increment; novelty is causal cosine from compatible cached vectors. No encoder/detector/SPI forward calls.


In [ ]:
import sys,os,json,time,platform,contextlib,traceback,datetime,hashlib,base64,zipfile,io,subprocess,shutil
from pathlib import Path
RUN=Path('/kaggle/working/experiments/stream_scheduler/runs')/datetime.datetime.now(datetime.timezone.utc).strftime('compact-real-cache-%Y%m%dT%H%M%SZ')
RUN.mkdir(parents=True,exist_ok=False)
(RUN/'logs').mkdir()
STATUS=[]
class Tee:
    def __init__(self,a,b):self.a,self.b=a,b
    def write(self,s):self.a.write(s);self.b.write(s);self.b.flush();return len(s)
    def flush(self):self.a.flush();self.b.flush()
@contextlib.contextmanager
def stage(name):
    start=time.perf_counter();row={'stage':name,'status':'running','started_utc':datetime.datetime.now(datetime.timezone.utc).isoformat()};STATUS.append(row)
    (RUN/'cell_status.json').write_text(json.dumps(STATUS,indent=2))
    with (RUN/'logs'/f'{name}.log').open('w',buffering=1) as log:
        with contextlib.redirect_stdout(Tee(sys.stdout,log)),contextlib.redirect_stderr(Tee(sys.stderr,log)):
            print('START',name,'output',RUN,flush=True)
            try:
                yield
                row['status']='complete'
            except BaseException as e:
                row['status']='failed';row['error']=type(e).__name__+': '+str(e);traceback.print_exc();raise
            finally:
                row['elapsed_seconds']=time.perf_counter()-start
                (RUN/'cell_status.json').write_text(json.dumps(STATUS,indent=2))
                print('END',name,row['status'],row['elapsed_seconds'],flush=True)
with stage('01_environment'):
    info={'interpreter':sys.executable,'python':sys.version,'platform':platform.platform(),'cpu_count':os.cpu_count(),'execution_target':'Kaggle managed CPU; submitted from WSL', 'gpu_requested':False,'dependencies':'Kaggle-managed torch/numpy/lightgbm; no notebook installations','run_dir':str(RUN)}
    (RUN/'environment.json').write_text(json.dumps(info,indent=2));print(json.dumps(info,indent=2),flush=True)


In [ ]:
with stage('02_source_snapshot'):
    blob=base64.b64decode('...')
    assert hashlib.sha256(blob).hexdigest()=='a40cc4df6cd00bd6d763d8296884a8d22c69107663bbdd089509c9f754342346'
    SOURCE=RUN/'source'
    SOURCE.mkdir()
    with zipfile.ZipFile(io.BytesIO(blob)) as z:
        z.extractall(SOURCE)
    expected={'Evaluation/compact_scene_transport.py': '15a7f1e4aac1435761de5ee53f8cc245c223fb057d281f588928d999fb95caf9', 'Evaluation/semantic_learning_env.py': '678842ba11f072be2a1ac319e6172c409fead870186c0c5aea864aa868308a0a', 'Scheduler/semantic_learning_policy.py': '4e7e13a7b7e311b35337c816dc710899cc2429109137514d94b0ed71f1fc91bc', 'Scheduler/training/semantic_learning_models.py': 'f7754cde828951e738e55b3727393b7c318f8d6dd560a1195c79b10d1f1d215d', 'experiments/stream_scheduler/runtime_bootstrap.py': 'e9a15e0545b7de1bd997a6b8d940c230cb3667447936e537166fdec4158d23ab', 'experiments/stream_scheduler/semantic_suite_data.py': '94215d5b6e3fb0b91e8e4beb838bd7cbc838efa87762e43134f512efe08a3e8a', 'experiments/stream_scheduler/train_real_suite.py': '1b4100b15066fca21d630b18e95ed0ccab76c3736e9ba2efdcf8e4a910d9d2b0', 'Test/tests/test_compact_scene_transport.py': 'c13149ed9f7c0e33cf3c39a40a1b6c7cae6e11d524c014998668a84cd50fcc70'}
    for name,value in expected.items():
        assert hashlib.sha256((SOURCE/name).read_bytes()).hexdigest()==value
    manifest={'commit':'032c23a3dafc7bfeb39a84ba55193b6b6f3a22a0','dirty_snapshot':True,'source_hashes':expected,'input_kernel':'satyapaladuguplus/semantic-image-pool-scene-cache-20261030/5','expected_archive_sha256':'0f8c6186b09dcf6ab87911df642612919377157bb1585082bed1c93b1506fe2c','gpu_budget':0,'internal_driver_timeout_seconds':10000}
    (RUN/'manifest.json').write_text(json.dumps(manifest,indent=2));print(json.dumps(manifest,indent=2),flush=True)


In [ ]:
with stage('03_input_and_regression'):
    matches=list(Path('/kaggle/input').rglob('Mini_dataset_6000.tar'))
    if len(matches)!=1:raise ValueError('Expected exactly one attached mini archive; found '+str(matches))
    ARCHIVE=matches[0];print('Input archive',ARCHIVE,'bytes',ARCHIVE.stat().st_size,flush=True)
    (RUN/'input.json').write_text(json.dumps({'path':str(ARCHIVE),'bytes':ARCHIVE.stat().st_size},indent=2))
    subprocess.run([sys.executable,'-m','unittest','discover','-s',str(SOURCE/'Test/tests'),'-p','test_compact_scene_transport.py','-v'],check=True,timeout=60)


In [ ]:
with stage('04_real_supervised_training'):
    import importlib.metadata as metadata
    versions={p:metadata.version(p) for p in ('torch','numpy','lightgbm')}
    print('Actual managed training packages',versions,flush=True)
    (RUN/'training_environment.json').write_text(json.dumps(versions,indent=2))
    command=[sys.executable,'-u',str(SOURCE/'experiments/stream_scheduler/train_real_suite.py'),'--archive',str(ARCHIVE),'--output',str(RUN/'results'),'--epochs','50','--seeds','1729,2718,31415','--training-episodes','64','--samples','512','--model-seconds','600']
    print('TRAIN COMMAND',command,flush=True)
    import threading
    with (RUN/'logs/driver.log').open('w',buffering=1) as log:
        proc=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        pump_errors=[]
        def forward_training_log():
            try:
                for line in proc.stdout:
                    log.write(line);log.flush()
                    print(line,end='',flush=True)
            except BaseException as error:
                pump_errors.append(repr(error))
        reader=threading.Thread(target=forward_training_log,daemon=True)
        reader.start()
        try:
            code=proc.wait(timeout=10000)
        except subprocess.TimeoutExpired:
            proc.kill();proc.wait();reader.join(timeout=10)
            raise RuntimeError('CPU training wall cap; partial checkpoints and logs retained')
        reader.join(timeout=10)
        if reader.is_alive() or pump_errors:raise RuntimeError('Log forwarding incomplete: '+str(pump_errors))
    if code:raise RuntimeError('Training driver failed '+str(code))
    print((RUN/'results/status.json').read_text(),flush=True)


In [ ]:
with stage('05_report_and_recovery'):
    summary=json.loads((RUN/'results/REPORT.json').read_text())
    text='# Real-cache scheduler component result\n\n'+json.dumps({k:v for k,v in summary.items() if k!='statuses'},indent=2)+'\n\nThis increment trains the listed supervised architectures on real cached image records with silver D reference targets; no independent urgency claim. RL and contextual VLM-P training are subsequent increments. Temporal missions/relevance/novelty and qualified teacher supervision remain the next components. This notebook did not alter image-cache or SPI production.\n'
    (RUN/'REPORT.md').write_text(text)
    print('Saved results, traces, source, manifest and cell logs under',RUN,flush=True)
# Capture finished cell status/logs after stage exit in the recovery ZIP.
paths={str(p.relative_to(RUN)):hashlib.sha256(p.read_bytes()).hexdigest() for p in RUN.rglob('*') if p.is_file() and p.name!='artifact_manifest.json' and '/logs/' not in str(p)}
(RUN/'artifact_manifest.json').write_text(json.dumps(paths,indent=2))
archive=shutil.make_archive(str(RUN.parent/(RUN.name+'-recovery')),'zip',RUN)
print('RECOVERY',archive,'sha256',hashlib.sha256(Path(archive).read_bytes()).hexdigest(),flush=True)
